In [ ]:
import obspy
from pathlib import Path


segydir = Path('~/Sarah_Mod17_SEGY/BXZ').expanduser()

print(segydir)



st = obspy.read(
    f"{segydir}/shot_001_BXZ_semv.segy",
    format="SEGY",
    unpack_trace_headers=True,
)


In [ ]:
import os 
os.getcwd()
os.chdir('/Users/glennthompson/Developer/karstgeo/01_modeling/specfem2d_split_refactored')

In [ ]:
from segy_tools.gather import plot_wiggle_gather_from_stream

plot_wiggle_gather_from_stream(
    st,
    tmin=0,
    tmax=0.15,
    omin=-75,
    omax=75,
    fallback_receiver_spacing_m=0.5,  # only used if SEG-Y headers lack coordinates
    fallback_source_x_m=150.0,        # only used if SEG-Y headers lack source x
    cave={"x_min_m": 145, "x_max_m": 155},
    outfile="shot_001_wiggle.png",
)

In [ ]:
import os
os.chdir('/Users/glennthompson/Developer/karstgeo/01_modeling/specfem2d')
os.getcwd()


In [ ]:
from segy_tools.io import read_su_file
from segy_tools.gather import plot_wiggle_gather_from_stream

outdir = Path("/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/02_Modelling/Seismic/specfem2d/glenn")

outdir_su_figs = outdir / "figures" / "su_wiggles"
outdir_su_figs.mkdir(parents=True, exist_ok=True)

files = sorted(Path("~/Downloads").expanduser().glob("*_*.su"))

for f in files:
    print(f)

    st = None
    last_error = None

    for byteorder in ("<", ">"):
        try:
            st = read_su_file(f, byteorder=byteorder)
            print(f"  read OK with byteorder={byteorder}")
            break
        except Exception as e:
            last_error = e
            print(f"  failed byteorder={byteorder}: {type(e).__name__}")

    if st is None:
        print(f"  skipping {f.name}: {last_error}")
        continue


    plot_wiggle_gather_from_stream(
        st,
        fallback_receiver_spacing_m=2.0,
        fallback_first_receiver_x_m=0.0,
        fallback_source_x_m=0.0,
        tmin=-0.05,
        tmax=0.25,
        scale=0.02,
        normalize=False,
        outfile=outdir_su_figs / f"{f.stem}_wiggle.png",
        title=f'{f.stem}: Model 15 highres mesh felix',
    )

In [ ]:
import numpy as np
import obspy
from pathlib import Path


def stream_to_gather_arrays_simple(
    st,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
):
    """
    Convert a single-component ObsPy Stream to time, data, receiver_x arrays.

    Assumes traces are already ordered by receiver/station. If SEG-Y headers
    contain geometry, we can improve this later by reading receiver x from headers.
    """
    st = st.copy()
    st.sort(keys=["station", "channel"])

    npts = min(tr.stats.npts for tr in st)
    dt = st[0].stats.delta

    data = np.vstack([
        tr.data[:npts].astype(float)
        for tr in st
    ])

    time = np.arange(npts) * dt + float(st[0].stats.starttime - st[0].stats.starttime)

    receiver_x_m = first_receiver_x_m + np.arange(len(st)) * receiver_spacing_m

    return time, data, receiver_x_m
def difference_segy_gathers(
    segy_a,
    segy_b,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
):
    """
    Read two SEG-Y files, match their dimensions, and return A, B, A-B arrays.
    """
    segy_a = Path(segy_a)
    segy_b = Path(segy_b)

    st_a = obspy.read(str(segy_a), format="SEGY", unpack_trace_headers=True)
    st_b = obspy.read(str(segy_b), format="SEGY", unpack_trace_headers=True)

    time_a, data_a, rx_a = stream_to_gather_arrays_simple(
        st_a,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
    )

    time_b, data_b, rx_b = stream_to_gather_arrays_simple(
        st_b,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
    )

    ntr = min(data_a.shape[0], data_b.shape[0])
    npts = min(data_a.shape[1], data_b.shape[1])

    time = time_a[:npts]
    receiver_x_m = rx_a[:ntr]

    data_a = data_a[:ntr, :npts]
    data_b = data_b[:ntr, :npts]

    diff = data_a - data_b

    return time, data_a, data_b, diff, receiver_x_m
def plot_model_difference_from_segy(
    model_a,
    model_b,
    segy_dir,
    component="BXZ",
    source_x_m=0.0,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    tmin=0.0,
    tmax=0.3,
    omin=None,
    omax=None,
    clip_percentile=98,
    outfile=None,
):
    """
    Difference two model SEG-Y gathers trace-by-trace and plot A, B, A-B.

    Expects files named like:
        Mod12_BXZ.segy
        Mod13_BXZ.segy
    """
    segy_dir = Path(segy_dir)

    segy_a = segy_dir / f"{model_a}_{component}.segy"
    segy_b = segy_dir / f"{model_b}_{component}.segy"

    time, data_a, data_b, diff, receiver_x_m = difference_segy_gathers(
        segy_a,
        segy_b,
        receiver_spacing_m=receiver_spacing_m,
        first_receiver_x_m=first_receiver_x_m,
    )

    fig = plot_difference_gathers(
        time=time,
        data_a=data_a,
        data_b=data_b,
        receiver_x_m=receiver_x_m,
        source_x_m=source_x_m,
        label_a=model_a,
        label_b=model_b,
        title=f"{model_a} - {model_b}, {component}",
        tmin=tmin,
        tmax=tmax,
        omin=omin,
        omax=omax,
        clip_percentile=clip_percentile,
        outfile=outfile,
    )

    return fig, diff
print(os.getcwd())
from segy_tools.plotting import plot_difference_gathers

segy_dir = outdir / "segy" / "specfem_models"
diff_fig_dir = outdir / "figures" / "specfem_model_differences"
diff_fig_dir.mkdir(parents=True, exist_ok=True)

fig, diff = plot_model_difference_from_segy(
    model_a="Mod12",
    model_b="Mod13",
    segy_dir=segy_dir,
    component="BXZ",
    source_x_m=0.0,
    receiver_spacing_m=1.0,
    first_receiver_x_m=0.0,
    tmin=0.0,
    tmax=0.3,
    clip_percentile=98,
    outfile=diff_fig_dir / "Mod12_minus_Mod13_BXZ.png",
)